In [6]:
import pubchempy as pcp
import pandas as pd
import time
import gzip
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import inchi

In [22]:
df_agro = pd.read_csv("../data/agro_data/PubChem_compound_CID__NXKSqAc4YoRVrmC34s8pkIwOI25vbjA-ShsrclEKOXNREwU.csv")
df = pd.read_csv("../data/nodes/SmallMolecule.csv")


In [8]:
def smiles_to_inchikey(smi: str):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    return inchi.MolToInchiKey(m)

df["inchikey"] = df["content"].astype(str).str.strip().apply(smiles_to_inchikey)

[21:05:14] WARNING: not removing hydrogen atom without neighbors
[21:05:14] WARNING: not removing hydrogen atom without neighbors
[21:06:14] bond type above 3 (12) is treated as unspecified!
[21:06:14] bond type above 3 (12) is treated as unspecified!
[21:06:14] Invalid InChI prefix in generating InChI Key
[21:06:26] bond type above 3 (12) is treated as unspecified!
[21:06:26] bond type above 3 (12) is treated as unspecified!
[21:06:26] bond type above 3 (12) is treated as unspecified!
[21:06:26] Invalid InChI prefix in generating InChI Key
[21:08:18] bond type above 3 (17) is treated as unspecified!
[21:08:18] Invalid InChI prefix in generating InChI Key
[21:11:39] bond type above 3 (17) is treated as unspecified!
[21:11:39] bond type above 3 (17) is treated as unspecified!
[21:11:39] bond type above 3 (17) is treated as unspecified!
[21:11:39] Invalid InChI prefix in generating InChI Key
[21:11:39] bond type above 3 (17) is treated as unspecified!
[21:11:39] bond type above 3 (17) is

In [23]:
our_set = set(df["inchikey"])
agro_set = set(df_agro["InChIKey"])
result_set = our_set & agro_set

In [24]:
res_df = df[df['inchikey'].isin(result_set)]
res_df

,content,name,alias,source,type,id,inchikey
10503,C/C=C(\C)C(=O)O[C@H]1C[C@@H](OC(C)=O)[C@@]2(C(...,AZADIRACTIN::CHEBI:2942,NaN,binding_db,SmallMolecule,83261,FTNJWQUOZFUQQJ-NDAWSKJSSA-N
10504,C/C=C(\C)C(=O)O[C@H]1C[C@@H](OC(C)=O)[C@@]2(C)...,Salannin,NaN,binding_db,SmallMolecule,83262,CJHBVBNPNXOWBA-REXVOHEDSA-N
14522,C=C(C)[C@H]1Cc2c(ccc3c2O[C@@H]2COc4cc(OC)c(OC)...,"(-)-cis-rotenone::(-)-rotenone::(2R,6aS,12aS)-...",NaN,binding_db,SmallMolecule,87280,JUVIOZPCNVVQFO-HBGVWJBISA-N
17172,C=C1CC[C@H](O)C/C1=C/C=C1\CCC[C@]2(C)[C@@H]([C...,CHEBI:28934::Calciferol::Calciferol In Arach O...,NaN,binding_db,SmallMolecule,89930,MECHNRXZTMCUDQ-RKHKHRCZSA-N
17175,C=C1CC[C@H](O)C/C1=C/C=C1\CCC[C@]2(C)[C@@H]([C...,7-Dehydrocholesterol::CHEBI:28940::Cholecalcif...,NaN,binding_db,SmallMolecule,89933,QYSXJUFSXHHAJI-YRZJJWOYSA-N
...,...,...,...,...,...,...,...
1298320,c1ccc2[nH]cnc2c1,"1H-1,3-benzodiazole::Benzimidazole::Benzimidaz...",NaN,binding_db,SmallMolecule,1371078,HYZJCKYKOHLVJF-UHFFFAOYSA-N
1299739,c1ccc2ccccc2c1,naphtaline::CHEMBL16293::naphtalene::naftalina...,CHEMBL16293::Naphthalen::Naphthalin::naftaleno...,binding_db,SmallMolecule,1372497,UFWIBTONFRDIAS-UHFFFAOYSA-N
1301342,c1coc(-c2nc3ccccc3[nH]2)c1,fuberidazole::2-(furan-2-yl)-1H-benzo[d]imidaz...,2-(furan-2-yl)-1H-benzo[d]imidazole::CHEMBL201...,binding_db,SmallMolecule,1374100,UYJUZNLFJAWNEZ-UHFFFAOYSA-N
1301409,c1coc(CNc2ncnc3[nH]cnc23)c1,"US9138393, Kinetin::MTH1 inhibitor, 88::US9144...","MTH1 inhibitor, 88::US9138393, Kinetin::US9144...",binding_db,SmallMolecule,1374167,QANMHLXAZMSUEX-UHFFFAOYSA-N


In [25]:
need = set(df["inchikey"].dropna().unique())

out = []  # (inchikey, cid)

with gzip.open("../data/pubchem/CID-InChI-Key.gz", "rt", encoding="utf-8", errors="ignore") as f:
    for line in f:
        parts = line.strip().split()

        cid = int(parts[0])
        inchikey = parts[-1]

        if inchikey in need:
            out.append((inchikey, cid))
            need.remove(inchikey)
            if not need:
                break

map_df = pd.DataFrame(out, columns=["inchikey", "cid"])
map_df

,inchikey,cid
0,VYZAHLCBVHPDDF-UHFFFAOYSA-N,6
1,MUIPLRMGAXZWSQ-UHFFFAOYSA-N,7
2,HERSSAVMHCMYSQ-UHFFFAOYSA-N,16
3,GLDQAMYCGOIJDV-UHFFFAOYSA-N,19
4,HWXBTNAVRSUOJR-UHFFFAOYSA-N,43
...,...,...
1242400,PJLOZUMADBELJZ-YAELJHOLSA-N,177668511
1242401,WYAXEPKCXNISBM-UHFFFAOYSA-N,177668512
1242402,SVNIJYDUCDBSDR-UHFFFAOYSA-N,177668513
1242403,BYMLLYZVHHLEBE-CQSZACIVSA-N,177668516


In [4]:
cid = 12345

assays = pcp.get_assays(cid, namespace="cid")
assays

BadRequestError: PubChem HTTP Error 400 PUGREST.BadRequest: Invalid identifier namespace (Expected identifier namespace: aid, listkey, type, sourceall, target, or activity)